In [2]:
import sys
sys.path.append("../")

import numpy as np
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from data_loader import PairData

# Real data — beta/intercept/std estimated from Q1
data = PairData("../../data/real_pair_MCD_YUM.csv")
n = len(data)
q1, q2 = n // 4, n // 2

jres      = coint_johansen(np.log(data.s[:, :q1]).T, det_order=0, k_ar_diff=1)
cv        = jres.evec[:, 0]
beta      = -cv[1] / cv[0]
intercept = float((np.log(data.s[0, :q1]) - beta * np.log(data.s[1, :q1])).mean())
spread_q1 = np.log(data.s[0, :q1]) - beta * np.log(data.s[1, :q1]) - intercept
fixed_std = float(spread_q1.std())

print(f"β={beta:.4f}  intercept={intercept:.4f}  std={fixed_std:.4f}")

# Load synthetic pairs
pairs = np.load("../../data/synthetic_MCD_YUM_pairs.npy")  # (N, 2, T)
N_SERIES = pairs.shape[0]
print(f"Loaded {N_SERIES} synthetic series of length {pairs.shape[2]}")

β=1.0437  intercept=0.5396  std=0.1157
Loaded 200 synthetic series of length 1258


In [3]:
%matplotlib widget

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import A2C
from env import Market

class SyntheticMarket(gym.Wrapper):
    """On each reset, samples a new random synthetic series."""
    def __init__(self, pairs, **market_kwargs):
        self._pairs = pairs
        self._market_kwargs = market_kwargs
        env = Market(self._make_pair(0), **market_kwargs)
        super().__init__(env)

    def _make_pair(self, idx):
        pd = object.__new__(PairData)
        pd.s = self._pairs[idx]  # (2, T)
        pd.labels = ["s0", "s1"]
        pd.timestamps = np.arange(self._pairs.shape[2])
        return pd

    def reset(self, **kwargs):
        idx = np.random.randint(len(self._pairs))
        self.env._pair_data = self._make_pair(idx)
        return self.env.reset(**kwargs)

class FlatMarket(gym.Wrapper):
    """Flattens dict obs to [z_score, entry_z, spread, entry_spread, position]."""
    def __init__(self, env):
        super().__init__(env)
        self.observation_space = spaces.Box(
            low= np.array([-np.inf, -np.inf, -np.inf, -np.inf, 0.0], dtype=np.float32),
            high=np.array([ np.inf,  np.inf,  np.inf,  np.inf, 1.0], dtype=np.float32),
        )

    def _flatten(self, obs):
        return np.array([
            obs["spread_z_score"][0],
            obs["entry_z_score"][0],
            obs["spread"][0],
            obs["entry_spread"][0],
            float(obs["position"]),
        ], dtype=np.float32)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        return self._flatten(obs), info

    def step(self, action):
        obs, reward, term, trunc, info = self.env.step(action)
        return self._flatten(obs), reward, term, trunc, info

train_env = FlatMarket(SyntheticMarket(
    pairs,
    fixed_beta=beta,
    spread_intercept=intercept,
    fixed_std=fixed_std,
    sparse_reward=True,
    transaction_cost=0.001,
    squared_reward=True,
))

TOTAL_STEPS = N_SERIES * pairs.shape[2] * 10  # 10 passes over all synthetic data
print(f"Training for {TOTAL_STEPS:,} steps across {N_SERIES} synthetic series...")

model = A2C("MlpPolicy", train_env, verbose=1, ent_coef=0.01, n_steps=50)
model.learn(total_timesteps=TOTAL_STEPS)
print("Done.")

Training for 2,516,000 steps across 200 synthetic series...
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 1.2e+03  |
|    ep_rew_mean        | 1.63e+06 |
| time/                 |          |
|    fps                | 2770     |
|    iterations         | 100      |
|    time_elapsed       | 1        |
|    total_timesteps    | 5000     |
| train/                |          |
|    entropy_loss       | -0.685   |
|    explained_variance | 7.75e-07 |
|    learning_rate      | 0.0007   |
|    n_updates          | 99       |
|    policy_loss        | -1.8e+05 |
|    value_loss         | 8.32e+10 |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 1.2e+03  |
|    ep_rew_mean        | 2.79e+06 |
| time/                 |          |
|    fps                | 2838  

KeyboardInterrupt: 

In [ ]:
from static_thresh_base import ZScoreBaseline
from runner import run_strategy, plot_strategy

class SB3Agent:
    def __init__(self, model):
        self.model = model

    def predict(self, obs):
        flat = np.array([
            obs["spread_z_score"][0],
            obs["entry_z_score"][0],
            obs["spread"][0],
            obs["entry_spread"][0],
            float(obs["position"]),
        ], dtype=np.float32)
        action, _ = self.model.predict(flat, deterministic=True)
        return int(action)

test_data  = data.slice(q2)
train_data = data.slice(q1, q2)
env_kwargs = dict(fixed_beta=beta, spread_intercept=intercept, fixed_std=fixed_std)

# A2C on real training data
results_train = run_strategy(Market(train_data, **env_kwargs), SB3Agent(model))
print("--- A2C (real train) ---")
plot_strategy(results_train)

# A2C on real test data
results_test = run_strategy(Market(test_data, **env_kwargs), SB3Agent(model))
print("--- A2C (real test) ---")
plot_strategy(results_test)

# Z-score baseline on test
results_base = run_strategy(Market(test_data, **env_kwargs), ZScoreBaseline(entry_threshold=2.0, exit_threshold=0.5))
print("--- Z-score baseline (test) ---")
plot_strategy(results_base)